In [1]:
import os
import datetime, pandas

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Union
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from abc import ABC, abstractmethod

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [2]:
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                         # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                         # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                # Multiplier of the OLS market forward excess returns predictions to signal 

# ============ RANDOM FOREST CONFIGS ============
N_ESTIMATORS: int = 100                         # Number of trees in the forest
MAX_DEPTH: int = None                           # Maximum depth of the tree
MIN_SAMPLES_SPLIT: int = 2                      # Minimum number of samples required to split an internal node
MIN_SAMPLES_LEAF: int = 1                       # Minimum number of samples required to be at a leaf node
RANDOM_STATE: int = 42                          # Random state for reproducibility

@dataclass
class DatasetOutput:
    X_train : pl.DataFrame 
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: StandardScaler

@dataclass 
class RandomForestParameters:
    n_estimators : int 
    max_depth: int
    min_samples_split: int
    min_samples_leaf: int
    random_state: int
    
    def __post_init__(self): 
        if self.n_estimators <= 0:
            raise ValueError("n_estimators must be greater than 0")
        
@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL

# Set the Parameters
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

rf_params = RandomForestParameters(
    n_estimators = N_ESTIMATORS,
    max_depth = MAX_DEPTH,
    min_samples_split = MIN_SAMPLES_SPLIT,
    min_samples_leaf = MIN_SAMPLES_LEAF,
    random_state = RANDOM_STATE
)

def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.
    """
    vars_to_keep = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]

    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in vars_to_keep
        ])
        .drop_nulls()
    )
    
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.
    """
    common_columns = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list) -> DatasetOutput: 
    """
    Splits the data into features (X) and target (y), and scales the features.
    """
    X_train = train.drop(['date_id','target']) 
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target']) 
    y_test = test.get_column('target')
    
    scaler = StandardScaler() 
    
    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)
    
    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)
    
    return DatasetOutput(
        X_train = X_train,
        y_train = y_train, 
        X_test = X_test, 
        y_test = y_test,
        scaler = scaler
    )

def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

# Looking at the Data
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset() 
print(train.tail(3)) 
print(test.head(3))

# Generating the Train and Test
df: pl.DataFrame = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df) 
train: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES = [col for col in test.columns if col not in ['date_id', 'target']]

dataset: DatasetOutput = split_dataset(train=train, test=test, features=FEATURES) 

X_train: pl.DataFrame = dataset.X_train
X_test: pl.DataFrame = dataset.X_test
y_train: pl.DataFrame = dataset.y_train
y_test: pl.DataFrame = dataset.y_test
scaler: StandardScaler = dataset.scaler 


shape: (3, 98)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────────────┬────────────────┬──────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9        ┆ forward_returns ┆ risk_free_rate ┆ target   │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---             ┆ ---            ┆ ---      │
│ i64     ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64       ┆ f64             ┆ f64            ┆ f64      │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════════════╪════════════════╪══════════╡
│ 8977    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.708599 ┆ 0.004187        ┆ 0.000162       ┆ 0.003713 │
│ 8978    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.725858 ┆ 0.002279        ┆ 0.000162       ┆ 0.001805 │
│ 8979    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.720092 ┆ 0.003541        ┆ 0.000161       ┆ 0.003068 │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────────────┴────────────────┴──────────┘
shape: (3, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scor

In [3]:
X_train = pandas.DataFrame(X_train)
X_test = pandas.DataFrame(X_test)
y_train = pandas.DataFrame(y_train)
y_test = pandas.DataFrame(y_test)

In [4]:
X_train

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,-0.285955,1.112088,1.674268,1.627662,0.166740,-0.129887,-0.821713,0.276656,0.635604,-0.138087,0.309654,-0.803976,-0.016005
1,-0.395515,1.123292,1.686506,1.629398,0.159627,-0.041275,-0.811496,0.305054,0.646199,-0.516780,0.308522,-0.794458,-0.014148
2,0.045640,1.144137,1.710228,1.631135,0.159589,-0.033100,-0.851134,0.312833,0.661098,-0.700966,0.307390,-0.842952,-0.014439
3,0.817345,1.141328,1.705303,1.632871,0.147790,-0.411799,-0.865743,0.306904,0.651978,-0.515000,0.306258,-0.853212,-0.015070
4,0.597401,1.131427,1.691863,1.634607,0.128952,-0.139996,-0.888585,0.303112,0.643282,-0.514195,0.305126,-0.872823,-0.014420
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7464,-0.112635,0.454874,0.853185,-0.869232,-0.024527,0.211188,1.205422,0.260464,0.726087,-0.620767,-0.345893,1.483810,-0.014590
7465,-0.050901,0.552017,0.955396,-0.851868,0.019529,0.080793,1.224751,0.326670,0.785761,-0.620042,0.607425,1.499570,-0.014406
7466,0.136916,0.521052,0.922062,-0.865759,0.019603,0.119403,1.182841,0.321577,0.769314,-0.619113,1.328640,1.442490,-0.015282
7467,0.198974,0.545914,0.947729,-0.853605,0.103247,0.092052,1.175003,0.330107,0.778347,-0.141948,0.666299,1.457698,-0.014689


In [5]:
X_test

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.071202,0.577252,0.979946,-0.884859,0.082628,-0.124857,1.193316,0.343976,0.791547,0.813018,1.002565,1.462864,-0.014660
1,0.259606,0.531720,0.931516,-0.874441,0.089617,0.052103,1.165067,0.324842,0.762751,-0.379850,0.717249,1.436112,-0.014608
2,0.321790,0.483057,0.879877,-0.877914,0.056102,0.227982,1.204837,0.302967,0.736316,0.345253,-0.990119,1.484485,-0.014716
3,-0.058333,0.514029,0.912258,-0.874441,0.016603,0.090467,1.244544,0.324163,0.750651,0.345485,0.544021,1.540940,-0.014149
4,-0.186197,0.566596,0.967275,-0.858814,0.036628,-0.021156,1.182242,0.345811,0.772634,0.598144,-0.599508,1.486083,-0.014469
5,-0.440632,0.542675,0.941613,-0.858814,0.050615,0.017770,1.160807,0.345010,0.760766,0.599486,0.422875,1.455728,-0.014867
6,-0.124694,0.553926,0.953023,-0.865759,0.085354,0.063544,1.098616,0.354122,0.765722,0.844027,0.813486,1.399496,-0.014670
7,-0.315846,0.569067,0.968520,-0.855341,0.137411,-0.057029,1.050089,0.355790,0.766012,-0.629474,1.052382,1.350650,-0.014263
8,-0.316883,0.586179,0.986030,-0.839714,0.166180,-0.032250,1.062804,0.355952,0.769013,1.335549,1.444125,1.363049,-0.014080
9,-0.000524,0.614430,1.015162,-0.837977,0.165597,-0.123211,1.034709,0.360814,0.789748,0.352787,1.316186,1.311524,-0.013962


In [6]:
improved_catboost_params = {'iterations': 3000,
                            'learning_rate': 0.01,
                            'depth': 6,
                            'l2_leaf_reg': 5.0,
                            'min_child_samples': 100,
                            'colsample_bylevel': 0.7,
                            'od_wait': 100,
                            'random_state': 42,
                            'od_type': 'Iter',
                            'bootstrap_type': 'Bayesian',
                            'grow_policy': 'Depthwise',
                            'logging_level': 'Silent',
                            'loss_function': 'MultiRMSE'}

R_Forest_parm = {'n_estimators': 100,
                 'min_samples_split': 5,
                 'max_depth': 15,
                 'min_samples_leaf': 3,
                 'max_features': 'sqrt',
                 'random_state': 42}
        
Extra_parm = {'n_estimators': 100,
              'min_samples_split': 5,
              'max_depth': 12,
              'min_samples_leaf': 3,
              'max_features': 'sqrt',
              'random_state': 42}
        
XGB_R_parm = {"n_estimators": 1500,
              "learning_rate": 0.05, 
              "max_depth": 6,
              "subsample": 0.8, 
              "colsample_bytree": 0.7,
              "reg_alpha": 1.0,
              "reg_lambda": 1.0,
              "random_state": 42}

LGBM_R_parm = {"n_estimators": 1500,
               "learning_rate": 0.05,
               "num_leaves": 50,
               "max_depth": 8,
               "reg_alpha": 1.0,
               "reg_lambda": 1.0,
               "random_state": 42,
               'verbosity': -1}

DecisionTree = {'criterion': 'poisson',
                'max_depth': 6}

GB_parm = {"learning_rate": 0.1,
           "min_samples_split": 500,
           "min_samples_leaf": 50,
           "max_depth": 8,
           "max_features": 'sqrt',
           "subsample": 0.8,
           "random_state": 10}

CatBoost = CatBoostRegressor(**improved_catboost_params)
XGBoost = XGBRegressor(**XGB_R_parm)
LGBM = LGBMRegressor(**LGBM_R_parm)
RandomForest = RandomForestRegressor(**R_Forest_parm)
ExtraTrees = ExtraTreesRegressor(**Extra_parm)
GBRegressor = GradientBoostingRegressor(**GB_parm)
        
DTRegressor = DecisionTreeRegressor(**DecisionTree)


estimators = [('CatBoost', CatBoost), ('XGBoost', XGBoost), ('LGBM', LGBM), ('RandomForest', RandomForest),
              ('ExtraTrees', ExtraTrees), ('GBRegressor', GBRegressor), 
              ('ElasticNet', ElasticNetCV()), ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
              ('LassoCV', LassoCV()), ('LassoLars', LassoLars())]

model = StackingRegressor(estimators, 
                          final_estimator = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]), 
                          cv=3)
model.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_stacking.py:957: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


StackingRegressor(cv=3,
                  estimators=[('CatBoost',
                               <catboost.core.CatBoostRegressor object at 0x7c7a4eb0b910>),
                              ('XGBoost',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.7, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            g...
                                                   min_samples_split=5,
                                                   random_state=42)),
                              ('GBRegressor',
                               GradientBoostingRegressor(max_depth=8,
                                                         max_features='sqrt',
                                                         min_samples_leaf=50,
                                                         min_samples_split=500,
                                                         random_state=10,
                                                         subsample=0.8)),
                              ('ElasticNet', ElasticNetCV()),
                              ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
                              ('LassoCV', LassoCV()),
                              ('LassoLars', LassoLars())],
                  final_estimator=RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]))

In [7]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({'lagged_forward_returns':'target'})
    df: pl.DataFrame = create_example_dataset(test)
    X_test: pl.DataFrame = pandas.DataFrame(df.select(FEATURES))
    X_test_scaled_np: np.ndarray = scaler.transform(X_test)
    X_test_scaled = pandas.DataFrame(pl.from_numpy(X_test_scaled_np, schema=FEATURES))
    raw_pred: float = model.predict(X_test_scaled)[0]
    return convert_ret_to_signal(raw_pred, ret_signal_params)

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but 